# MandiGuard Standalone Google Colab Reproduction (< 3 Minutes)

Upload **`colab_data_pack.tar.gz`** (<10 MB) alongside this notebook in Google Colab and click **Run all**.
It unpacks the self-contained 2024 pre-trained model, stratified 2025-2026 holdout panel (~13.9k episodes), complete 2026 HP Apple harvest panel, verified GoI survey parameters, and source modules—reproducing the main benchmark, Grade-Explicit HP Apple evaluation, tool ablations, and `R=1,000` cluster bootstrap cleanly in under 3 minutes.


In [1]:
import json, os, pickle, sys, tarfile, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

t0 = time.time()
pack_candidates = [
    Path("colab_data_pack.tar.gz"),
    Path("../data/processed/colab_data_pack.tar.gz"),
    Path("data/processed/colab_data_pack.tar.gz"),
]
pack_path = next((p for p in pack_candidates if p.exists()), None)
if pack_path is None:
    raise FileNotFoundError("Please upload colab_data_pack.tar.gz to your working directory before running.")

work_dir = Path("mandiguard_colab_work")
work_dir.mkdir(exist_ok=True)
with tarfile.open(pack_path, "r:gz") as tar:
    tar.extractall(work_dir)
sys.path.insert(0, str(work_dir.resolve()))
print(f"Unpacked {pack_path.name} in {time.time()-t0:.2f}s")


Unpacked colab_data_pack.tar.gz in 0.04s


In [2]:
from src.agents.mandiguard_agent import AgentConfig, evaluate_mandiguard_vectorized
from src.baselines.dispatch_baselines import eval_b1_naive_screen_greedy, eval_b2_fresh_road_greedy, eval_b3_ml_point_dispatch, eval_b4_upfront_all_tools
from src.evaluation.bootstrap import two_way_mandi_week_bootstrap, holm_bonferroni

cfg = json.loads((work_dir / "colab_data/config.json").read_text())
cal = json.loads((work_dir / "colab_data/frozen_calibration.json").read_text())
frozen_margin = float(cal["frozen_safety_margin_inr"])
hurdle = float(cfg["agent_defaults"]["screen_hurdle_inr_qtl"])

with open(work_dir / "colab_data/forecaster_2024.pkl", "rb") as f:
    forecaster = pickle.load(f)

hold_df = pd.read_parquet(work_dir / "colab_data/holdout_sample_6state.parquet")
p_ret, l_ret = forecaster.predict_point_and_causal_lcb(hold_df)
hold_df["pred_ret"], hold_df["lcb_ret"] = p_ret, l_ret

acfg = AgentConfig(screen_hurdle_inr=hurdle, k_max=5, safety_margin_inr=frozen_margin)
mg, _ = evaluate_mandiguard_vectorized(hold_df, acfg)
b1 = eval_b1_naive_screen_greedy(hold_df, hurdle_inr=hurdle)
b2 = eval_b2_fresh_road_greedy(hold_df, hurdle_inr=hurdle)
b3 = eval_b3_ml_point_dispatch(hold_df, safety_margin_inr=frozen_margin)
b4 = eval_b4_upfront_all_tools(hold_df, acfg)

mg_zc, _ = evaluate_mandiguard_vectorized(
    hold_df, AgentConfig(screen_hurdle_inr=hurdle, k_max=5, safety_margin_inr=frozen_margin, zerocloud_mode=True)
)
mg_sat, _ = evaluate_mandiguard_vectorized(
    hold_df, AgentConfig(screen_hurdle_inr=hurdle, k_max=5, safety_margin_inr=frozen_margin, zerocloud_mode=True, early_exit_satisficing_inr=250.0)
)
from src.evaluation.payoff_model import PayoffParams
from src.evaluation.run_creative_decade_and_modern_experiments import _exact_vectorized_freight_shift
hold_fpo = _exact_vectorized_freight_shift(hold_df, 0.70)
mg_fpo, _ = evaluate_mandiguard_vectorized(hold_fpo, acfg, payoff_params=PayoffParams(freight_mult=0.70, use_state_statutory_fees=True))

rows = []
for name, d in [
    ("B1_NaiveScreen", b1),
    ("B2_FreshRoad", b2),
    ("B3_MLPoint", b3),
    ("B4_UpfrontAll", b4),
    ("MandiGuard_Standard", mg),
    ("MandiGuard_ZeroCloud", mg_zc),
    ("MandiGuard_ZeroCloud_Sat250", mg_sat),
    ("MandiGuard_FPO_90qtl_0.7x", mg_fpo),
]:
    disp = d["is_dispatch"]
    rows.append({
        "policy": name,
        "episodes": len(d),
        "dispatch_pct": round(float(disp.mean() * 100), 2),
        "mean_net_gain_inr": round(float(d["delta_net_inr"].mean()), 2),
        "loss_trip_pct": round(float(d.loc[disp, "is_phantom_loss"].mean() * 100), 2),
        "closed_trap_pct": round(float(d.loc[disp, "is_closed_trap"].mean() * 100), 2),
        "live_api_calls": round(float(d["tool_calls_total"].mean()), 2),
    })
print("=== Stratified Holdout Reproduction (incl. ZeroCloud & FPO Pooling) ===")
print(pd.DataFrame(rows).to_string(index=False))


=== Stratified Holdout Reproduction (incl. ZeroCloud & FPO Pooling) ===
                     policy  episodes  dispatch_pct  mean_net_gain_inr  loss_trip_pct  closed_trap_pct  live_api_calls
             B1_NaiveScreen     13906         89.98             337.21          40.85            59.50            0.00
               B2_FreshRoad     13906         61.72             433.61          17.88            24.14            7.85
                 B3_MLPoint     13906         92.77             318.66          45.37            66.00            7.85
              B4_UpfrontAll     13906         47.29             446.57           9.67            16.50           15.69
        MandiGuard_Standard     13906         47.29             446.57           9.67            16.50            8.70
       MandiGuard_ZeroCloud     13906         47.29             446.57           9.67            16.50            5.48
MandiGuard_ZeroCloud_Sat250     13906         47.29             433.51          10.02          

In [3]:
# Reproduce Cluster Bootstrap & Inspect Full-Panel + Decade/Edge Reference Results
ep_meta = hold_df[["episode_id", "origin_market_key", "date"]].drop_duplicates("episode_id").copy()
for k, dfp in [("B1", b1), ("B2", b2), ("B3", b3), ("B4", b4), ("MG", mg)]:
    m = dfp.set_index("episode_id")
    ep_meta[f"{k}_dnet"] = ep_meta["episode_id"].map(m["delta_net_inr"])
    ep_meta[f"{k}_pl"] = ep_meta["episode_id"].map(m["is_phantom_loss"]).astype(float)
    ep_meta[f"{k}_calls"] = ep_meta["episode_id"].map(m["tool_calls_total"]).astype(float)
ep_meta["B4_75"] = 0.75 * ep_meta["B4_calls"]

tests = [
    ("H1_MG_gt_B1_dNet", "MG_dnet", "B1_dnet", "greater"),
    ("H2_MG_gt_B2_dNet", "MG_dnet", "B2_dnet", "greater"),
    ("H3_MG_lt_B3_Phantom", "MG_pl", "B3_pl", "less"),
    ("H4_MG_lt_75pct_B4_Calls", "MG_calls", "B4_75", "less"),
]
b_rows = []
raw_ps = []
for tid, c1, c2, dr in tests:
    res = two_way_mandi_week_bootstrap(ep_meta, c1, c2, reps=1000, seed=20261117, direction=dr)
    raw_ps.append(res["p_value"])
    b_rows.append({"test": tid, "estimate": round(res["estimate"], 4), "ci95": [round(res["ci95_low"], 4), round(res["ci95_high"], 4)], "p_raw": res["p_value"]})
hps = holm_bonferroni(raw_ps)
for i, r in enumerate(b_rows):
    r["p_holm"] = round(hps[i], 5)
print("\n=== Colab Cluster Bootstrap Summary (R=1,000) ===")
print(pd.DataFrame(b_rows).to_string(index=False))

print("\n=== Full-Panel Reference Benchmark (69,526 Holdout Episodes) ===")
print(pd.read_csv(work_dir / "reference_results/exp2_main_benchmark_6state.csv").to_string(index=False))
print("\n=== ZeroCloud Edge-AI & FPO Pooling Full-Panel Reference ===")
print(pd.read_csv(work_dir / "reference_results/exp12_zerocloud_money_efficient_agent.csv").to_string(index=False))
print(pd.read_csv(work_dir / "reference_results/exp13_outofbox_fpo_pooling_and_spoilage_clock.csv").to_string(index=False))
print(f"\nTotal Colab Standalone Elapsed Time: {time.time()-t0:.2f}s")



=== Colab Cluster Bootstrap Summary (R=1,000) ===
                   test  estimate                ci95    p_raw  p_holm
       H1_MG_gt_B1_dNet  109.3634 [48.9864, 169.8872] 0.000999 0.00400
       H2_MG_gt_B2_dNet   12.9635 [-15.3397, 39.4235] 0.187812 0.18781
    H3_MG_lt_B3_Phantom   -0.3752  [-0.4073, -0.3431] 0.000999 0.00400
H4_MG_lt_75pct_B4_Calls   -3.0667   [-3.4488, -2.709] 0.000999 0.00400

=== Full-Panel Reference Benchmark (69,526 Holdout Episodes) ===
                  policy  n_episodes  dispatch_rate_pct  mean_delta_net_inr  cond_dispatched_delta_net_inr  open_d0_mean_delta_net_inr  ep_phantom_loss_pct  cond_phantom_loss_rate_pct  cond_closed_or_trap_rate_pct  cvar_10_inr  avg_tool_calls  lot_30qtl_gain_inr
            B0_SellLocal       69526               0.00                0.00                           0.00                        0.00                 0.00                        0.00                          0.00         0.00            0.00                 0.0
  